In [2]:
!pip install google-cloud-bigquery pandas scipy statsmodels --quiet

from google.cloud import bigquery
from google.colab import auth
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

auth.authenticate_user()
project_id = "project-1247b00d-348a-45d2-be6"
client = bigquery.Client(project=project_id)

In [3]:
query = """
SELECT *
FROM `dual_burden_diabetes_depression.analysis_cohort`
WHERE phq9_score IS NOT NULL AND hba1c IS NOT NULL
"""

df = client.query(query).to_dataframe()

# create the comorbidity flag we'll use throughout
df["comorbid"] = df["phq9_score"] >= 10

print(f"Rows pulled: {len(df)}")
print(df["comorbid"].value_counts())

Rows pulled: 735
comorbid
False    642
True      93
Name: count, dtype: int64


In [4]:
outcomes = ["hba1c", "routine_visits_est", "hospital_nights_est"]

for outcome in outcomes:
    group_comorbid = df[df["comorbid"] == True][outcome].dropna()
    group_noncomorbid = df[df["comorbid"] == False][outcome].dropna()

    t_stat, p_val = stats.ttest_ind(group_comorbid, group_noncomorbid, equal_var=False)

    print(f"{outcome}:")
    print(f"  comorbid mean = {group_comorbid.mean():.2f}, non-comorbid mean = {group_noncomorbid.mean():.2f}")
    print(f"  t = {t_stat:.2f}, p = {p_val:.4f}")
    print()

hba1c:
  comorbid mean = 7.87, non-comorbid mean = 7.28
  t = 2.75, p = 0.0071

routine_visits_est:
  comorbid mean = 6.38, non-comorbid mean = 5.54
  t = 1.62, p = 0.1090

hospital_nights_est:
  comorbid mean = 0.45, non-comorbid mean = 0.30
  t = 1.39, p = 0.1672



In [5]:
query_secondary = """
SELECT *
FROM `dual_burden_diabetes_depression.analysis_cohort`
WHERE phq9_score IS NOT NULL AND hba1c IS NOT NULL AND ldl_mmol IS NOT NULL
"""

df_secondary = client.query(query_secondary).to_dataframe()
df_secondary["comorbid"] = df_secondary["phq9_score"] >= 10

group_comorbid = df_secondary[df_secondary["comorbid"] == True]["ldl_mmol"].dropna()
group_noncomorbid = df_secondary[df_secondary["comorbid"] == False]["ldl_mmol"].dropna()

t_stat, p_val = stats.ttest_ind(group_comorbid, group_noncomorbid, equal_var=False)

print(f"LDL (mmol/L):")
print(f"  comorbid mean = {group_comorbid.mean():.2f}, non-comorbid mean = {group_noncomorbid.mean():.2f}")
print(f"  t = {t_stat:.2f}, p = {p_val:.4f}")

LDL (mmol/L):
  comorbid mean = 2.59, non-comorbid mean = 2.48
  t = 0.55, p = 0.5870


In [6]:
model = smf.ols("hba1c ~ comorbid + age + sex", data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  hba1c   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     5.473
Date:                Fri, 28 Aug 2026   Prob (F-statistic):            0.00101
Time:                        20:15:34   Log-Likelihood:                -1366.3
No. Observations:                 735   AIC:                             2741.
Df Residuals:                     731   BIC:                             2759.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            7.8665      0.361  

In [7]:
model_visits = smf.ols("routine_visits_est ~ comorbid + age + sex", data=df).fit()
print(model_visits.summary())

                            OLS Regression Results                            
Dep. Variable:     routine_visits_est   R-squared:                       0.016
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     3.902
Date:                Fri, 28 Aug 2026   Prob (F-statistic):            0.00880
Time:                        20:44:09   Log-Likelihood:                -2095.3
No. Observations:                 733   AIC:                             4199.
Df Residuals:                     729   BIC:                             4217.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            2.9605      0.982  

In [8]:
model_nights = smf.ols("hospital_nights_est ~ comorbid + age + sex", data=df).fit()
print(model_nights.summary())

                             OLS Regression Results                            
Dep. Variable:     hospital_nights_est   R-squared:                       0.013
Model:                             OLS   Adj. R-squared:                  0.009
Method:                  Least Squares   F-statistic:                     3.280
Date:                 Fri, 28 Aug 2026   Prob (F-statistic):             0.0205
Time:                         20:44:43   Log-Likelihood:                -869.32
No. Observations:                  733   AIC:                             1747.
Df Residuals:                      729   BIC:                             1765.
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -0.0870   

In [9]:
model_ldl = smf.ols("ldl_mmol ~ comorbid + age + sex", data=df_secondary).fit()
print(model_ldl.summary())

                            OLS Regression Results                            
Dep. Variable:               ldl_mmol   R-squared:                       0.065
Model:                            OLS   Adj. R-squared:                  0.057
Method:                 Least Squares   F-statistic:                     8.493
Date:                Fri, 28 Aug 2026   Prob (F-statistic):           1.81e-05
Time:                        20:48:27   Log-Likelihood:                -510.98
No. Observations:                 370   AIC:                             1030.
Df Residuals:                     366   BIC:                             1046.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            3.2141      0.317  

In [10]:
import os

os.makedirs("results", exist_ok=True)

# 1. Descriptive summary tables (primary cohort)
summary_primary = df.groupby("comorbid")[["hba1c", "routine_visits_est", "hospital_nights_est"]].agg(["mean", "std", "count"])
summary_primary.to_csv("results/summary_primary_cohort.csv")

# 2. Descriptive summary table (secondary/LDL cohort)
summary_secondary = df_secondary.groupby("comorbid")[["ldl_mmol"]].agg(["mean", "std", "count"])
summary_secondary.to_csv("results/summary_secondary_cohort.csv")

# 3. Final results table — unadjusted + adjusted, all four outcomes
results_table = pd.DataFrame([
    {"outcome": "hba1c", "cohort_n": 735, "unadjusted_p": 0.0071, "adjusted_p": 0.001, "adjusted_coef": 0.5718, "adjusted_ci_low": 0.232, "adjusted_ci_high": 0.912},
    {"outcome": "routine_visits_est", "cohort_n": 733, "unadjusted_p": 0.1090, "adjusted_p": 0.055, "adjusted_coef": 0.9075, "adjusted_ci_low": -0.020, "adjusted_ci_high": 1.835},
    {"outcome": "hospital_nights_est", "cohort_n": 733, "unadjusted_p": 0.1672, "adjusted_p": 0.068, "adjusted_coef": 0.1623, "adjusted_ci_low": -0.012, "adjusted_ci_high": 0.337},
    {"outcome": "ldl_mmol", "cohort_n": 370, "unadjusted_p": 0.5870, "adjusted_p": 0.772, "adjusted_coef": 0.0422, "adjusted_ci_low": -0.244, "adjusted_ci_high": 0.328},
])
results_table.to_csv("results/final_results_table.csv", index=False)

# 4. The full analysis-ready dataset itself (primary cohort, for Power BI)
df.to_csv("results/analysis_cohort_primary.csv", index=False)
df_secondary.to_csv("results/analysis_cohort_secondary.csv", index=False)

print("All files saved to /results")

All files saved to /results


In [11]:
import shutil
from google.colab import files

shutil.make_archive("results_export", "zip", "results")
files.download("results_export.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>